# 05 Equal learning rate factor (exploratory sensitivity analysis, added after the protocol)
The selection runs gave Qp4 and Q8 the factor k = 1, so their block weights barely moved, while Q4 and Qp8 used k = 100.
This notebook retrains Qp4 (plan A) and Q8 (plan B) with k = 100, with the same folds, seeds, initial weights and code as the
main runs, so that Q4 vs Qp4 and Q8 vs Qp8 compare blocks trained with the same factor. Outputs go to `runs_k100/` only;
`runs/` is not touched. T4 GPU only. Two sessions can run at the same time (set `SESSION = 'B'` in the second one).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, time
ROOT = '/content/drive/MyDrive/BUSI_QB'
CODE = os.path.join(ROOT, 'code'); PREP = os.path.join(ROOT, 'prepared')
RUNS_K = os.path.join(ROOT, 'runs_k100'); os.makedirs(RUNS_K, exist_ok=True)
sys.path.insert(0, os.path.join(CODE, 'src'))
SESSION = 'A'                                     # 'B' in a second session
import numpy as np, pandas as pd, tensorflow as tf
import qsim, qbus, runner
print(qbus.env_info())
assert 'T4' in str(qbus.env_info().get('gpu')), 'select a T4 GPU runtime'
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print('determinism not available:', e)
K = 100
todo = [dict(plan='A', model='qp4', fold=f, seed=s, frac=1.0, k=K) for s in runner.SEEDS for f in range(5)]
todo += [dict(plan='B', model='q8', fold=f, seed=s, frac=1.0, k=K) for s in runner.SEEDS for f in range(5)]
todo = [r for pair in zip(todo[:25], todo[25:]) for r in pair]      # alternate Qp4 and Q8
done = glob.glob(os.path.join(RUNS_K, '*', '*', 'summary.json'))
print(len(todo), 'runs in total,', len(done), 'done')

In [ ]:
# 2. RUNS ----------------------------------------------------------------------------------------------
for r in runner.ordered(todo, SESSION):
    root = os.path.join(RUNS_K, r['plan']); os.makedirs(root, exist_ok=True)
    rid = runner.run_id(r)
    if os.path.exists(os.path.join(root, rid, 'summary.json')):
        continue
    if runner.claim(root, rid, SESSION):
        print(time.strftime('%m-%d %H:%M'), 'start', rid, 'k =', r['k'])
        s = runner.one_run(r, PREP, root, session=SESSION)
        print('   Dice', round(s['test_dice'], 4), 'k', s['k'], 'block change', round(s.get('change_block', float('nan')), 4),
              'min', round(s['minutes'], 1))

In [ ]:
# 3. COLLECT (when all 50 runs are done) -----------------------------------------------------------------
dirs = sorted(glob.glob(os.path.join(RUNS_K, '*', '*')))
res = []
for d in dirs:
    s = os.path.join(d, 'summary.json')
    if not os.path.exists(s):
        continue
    out = dict(summary=json.load(open(s)))
    for nm in ('test_busi', 'test_busbra', 'test_breast', 'normals_busi', 'code_replacement'):
        f = os.path.join(d, nm + '.csv')
        if os.path.exists(f):
            t = pd.read_csv(f); t.insert(0, 'run_id', out['summary']['run_id']); out[nm] = t
    res.append(out)
print(f'done {len(res)} / 50, running {len(glob.glob(os.path.join(RUNS_K, "*", "*.lock")))}')
if len(res) == 50:
    OUT = os.path.join(ROOT, 'analysis', 'k100'); os.makedirs(OUT, exist_ok=True)
    summ = pd.DataFrame([{k: (json.dumps(v) if isinstance(v, (dict, list)) else v) for k, v in r['summary'].items()} for r in res])
    summ.to_csv(os.path.join(OUT, 'summaries.csv'), index=False)
    for nm in ('test_busi', 'test_busbra', 'test_breast', 'normals_busi', 'code_replacement'):
        parts = [r[nm] for r in res if nm in r]
        if parts:
            t = pd.concat(parts, ignore_index=True)
            for c in t.columns:
                if t[c].dtype == float:
                    t[c] = t[c].round(5)
            t.to_csv(os.path.join(OUT, nm + '.csv.gz'), index=False, compression='gzip')
            print(nm, len(t), 'rows')
    print(summ.groupby('model').agg(runs=('run_id', 'size'), k=('k', 'first'), dice=('test_dice', 'mean'),
                                     change=('change_block', 'median')).round(4).to_string())
    print('K100 COLLECT OK')